# College Football Recruiting and Team Success

## Research Question

**What is the relationship between a college football team's recruiting strength and its on-field success among FBS teams?**

A related question is:

**Do teams with higher-rated recruiting classes tend to have higher winning percentages and stronger overall team ratings?**

This project uses College Football Data API information from the **2021 through 2025** seasons.


## 1. Setup

This notebook uses Python, pandas, requests, and matplotlib.

**Important:** Do not paste your CFBD API key directly into this notebook if you plan to upload the notebook to GitHub.


In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 2. Load the CFBD API key

Before running this cell, set an environment variable named `CFBD_API_KEY`.

In a Windows PowerShell terminal, use:

```powershell
$env:CFBD_API_KEY="YOUR_REAL_KEY_HERE"
```

Then return to this notebook and run the next cell.


In [ ]:
API_KEY = os.getenv("CFBD_API_KEY")

if API_KEY:
    print("API key loaded successfully.")
else:
    print("API key NOT found. Set CFBD_API_KEY in your terminal before continuing.")

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

base_url = "https://api.collegefootballdata.com"
seasons = [2021, 2022, 2023, 2024, 2025]


## 3. Team Records

We will pull season records for FBS teams and calculate winning percentage.


In [ ]:
records_list = []

for year in seasons:
    response = requests.get(
        f"{base_url}/records",
        headers=headers,
        params={"year": year}
    )
    response.raise_for_status()

    for team in response.json():
        if str(team.get("classification", "")).lower() == "fbs":
            total = team.get("total", {})
            records_list.append({
                "season": team.get("year"),
                "team": team.get("team"),
                "conference": team.get("conference"),
                "games": total.get("games"),
                "wins": total.get("wins"),
                "losses": total.get("losses"),
                "ties": total.get("ties")
            })

records_df = pd.DataFrame(records_list)
records_df.head()


In [ ]:
print("Records shape:", records_df.shape)
display(records_df.head())
display(records_df.isnull().sum())
print("Duplicate rows:", records_df.duplicated().sum())


In [ ]:
records_df["winning_percentage"] = records_df["wins"] / records_df["games"]
records_df.head()


## 4. Recruiting Data

Recruiting strength will be measured using team recruiting ranking and recruiting points from the College Football Data API.


In [ ]:
recruiting_list = []

for year in seasons:
    response = requests.get(
        f"{base_url}/recruiting/teams",
        headers=headers,
        params={"year": year}
    )
    response.raise_for_status()

    for row in response.json():
        recruiting_list.append(row)

recruiting_raw = pd.DataFrame(recruiting_list)

print("Recruiting columns:")
print(recruiting_raw.columns.tolist())
display(recruiting_raw.head())


In [ ]:
# Standardize the recruiting columns.
# CFBD commonly returns: year, rank, team, points.

recruiting_df = recruiting_raw.rename(columns={
    "year": "season",
    "rank": "recruiting_rank",
    "points": "recruiting_score"
})

wanted_cols = ["season", "team", "recruiting_rank", "recruiting_score"]
recruiting_df = recruiting_df[[c for c in wanted_cols if c in recruiting_df.columns]].copy()

display(recruiting_df.head())
print("Recruiting shape:", recruiting_df.shape)
display(recruiting_df.isnull().sum())


## 5. Merge Recruiting and Team Records

Each row in the merged dataset should represent one **team-season**.


In [ ]:
df = pd.merge(
    records_df,
    recruiting_df,
    on=["season", "team"],
    how="inner"
)

print("Merged shape:", df.shape)
display(df.head())
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())


## 6. SP+ Team Ratings

SP+ gives us another measure of team strength to compare with recruiting.


In [ ]:
sp_list = []

for year in seasons:
    response = requests.get(
        f"{base_url}/ratings/sp",
        headers=headers,
        params={"year": year}
    )
    response.raise_for_status()

    for row in response.json():
        sp_list.append({
            "season": row.get("year"),
            "team": row.get("team"),
            "sp_rating": row.get("rating")
        })

sp_df = pd.DataFrame(sp_list)

print("SP+ shape:", sp_df.shape)
display(sp_df.head())
display(sp_df.isnull().sum())


In [ ]:
df = pd.merge(
    df,
    sp_df,
    on=["season", "team"],
    how="left"
)

print("Final analysis dataset shape:", df.shape)
display(df.head())
display(df.isnull().sum())


## 7. Data Cleaning

We will remove rows missing the variables required for each analysis.

We keep the original merged dataset in `df` and create a cleaned copy for the main visualizations.


In [ ]:
analysis_df = df.dropna(
    subset=["recruiting_score", "winning_percentage"]
).copy()

print("Rows before cleaning:", len(df))
print("Rows after cleaning for main analysis:", len(analysis_df))
print("Rows removed:", len(df) - len(analysis_df))


## 8. Visualization 1: Recruiting Strength vs. Winning Percentage


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    analysis_df["recruiting_score"],
    analysis_df["winning_percentage"],
    alpha=0.6
)

plt.xlabel("Recruiting Score")
plt.ylabel("Winning Percentage")
plt.title("Recruiting Strength vs. Winning Percentage, 2021–2025")
plt.grid(alpha=0.2)
plt.show()


### Interpretation

After you run the graph, write 2–4 sentences here describing the overall pattern you see.

Things to look for:
- Does winning percentage generally rise as recruiting score rises?
- Are there major outliers?
- Is the relationship weak, moderate, or strong?


## 9. Visualization 2: Recruiting Strength vs. SP+ Rating


In [ ]:
sp_analysis_df = df.dropna(
    subset=["recruiting_score", "sp_rating"]
).copy()

plt.figure(figsize=(8, 6))
plt.scatter(
    sp_analysis_df["recruiting_score"],
    sp_analysis_df["sp_rating"],
    alpha=0.6
)

plt.xlabel("Recruiting Score")
plt.ylabel("SP+ Rating")
plt.title("Recruiting Strength vs. SP+ Rating, 2021–2025")
plt.grid(alpha=0.2)
plt.show()


### Interpretation

After you run the graph, write 2–4 sentences here describing what the visualization shows.


## 10. Correlations

Correlation gives us a numerical summary of the relationship between recruiting strength and the two performance measures.


In [ ]:
corr_win = analysis_df["recruiting_score"].corr(
    analysis_df["winning_percentage"]
)

corr_sp = sp_analysis_df["recruiting_score"].corr(
    sp_analysis_df["sp_rating"]
)

print("Recruiting Score vs Winning Percentage correlation:", round(corr_win, 3))
print("Recruiting Score vs SP+ Rating correlation:", round(corr_sp, 3))


## 11. Final Dataset Summary

Use the outputs below to fill in the dataset-size and missing-value sections of your website write-up.


In [ ]:
print("FINAL DATASET ROWS:", df.shape[0])
print("FINAL DATASET COLUMNS:", df.shape[1])

print("\nMissing values by column:")
print(df.isnull().sum())

print("\nColumns:")
print(df.columns.tolist())


## 12. Save the Cleaned Data


In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)

df.to_csv(
    "data/college_football_recruiting_success.csv",
    index=False
)

print("Saved: data/college_football_recruiting_success.csv")


## 13. Conclusion

After running the analysis, write a short conclusion here that answers the research question using the actual graphs and correlation values.

Be careful not to claim that recruiting **causes** success. This is an observational analysis showing association.


## 14. AI Usage Disclosure

I used OpenAI ChatGPT (GPT-5.6) to help interpret the assignment requirements, organize the notebook, troubleshoot Python code, and improve explanations of analytical decisions. I reviewed the suggestions and am responsible for the final code, analysis, visualizations, and written content included in this project.
